# Neural network classification: SM vs BSM

Train a feedforward classifier on kinematic observables from `ForAnalysis/1d`. **This notebook uses one representative BSM operator (chbtil)** for the main training loop; the full six-operator sweep is produced by repeating the workflow or via the saved checkpoints in `models/`.


In [ ]:
import pandas as pd
import numpy as np
import h5py
import matplotlib.pyplot as plt
from plot_style import apply_publication_style
apply_publication_style()
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Load Data

In [ ]:
# Load SM + BSM datasets (this notebook trains SM vs chbtil only)
def load_dataset(filepath):
    with h5py.File(filepath, 'r') as f:
        df = pd.DataFrame.from_records(f['ForAnalysis/1d'][:])
    return df

sm_df = load_dataset('datasets/new_Input_bbyy_SMEFT_SM_4thMarch_2026.h5')
chbtil_df = load_dataset('datasets/new_Input_bbyy_SMEFT_chbtil_4thMarch_2026.h5')

print(f"SM events: {len(sm_df):,}")
print(f"chbtil events: {len(chbtil_df):,}")

## 2. Prepare Features

In [ ]:
# Define input features (exclude metadata columns)
exclude_columns = ['EventNumber', 'is_HHEvent', 'is_SingleHiggsEvent', 'is_SingleZEvent', 'is_ZHEvent', 'Lumi_weight', 'nBTaggedJets', 'NJets']
feature_columns = [col for col in sm_df.columns if col not in exclude_columns]

print(f"Number of features: {len(feature_columns)}")
print(f"Features: {feature_columns}")

In [ ]:
# Prepare data for binary classification: SM (0) vs BSM (1)
# `TRAIN_BSM_OPERATORS` + `bsm_datasets` control which operators are trained (chbtil only here).

def prepare_binary_data(sm_df, bsm_df, feature_columns, balance_classes=True):
    """
    Prepare data for SM vs BSM classification.
    
    Parameters:
    - sm_df: Standard Model DataFrame
    - bsm_df: BSM DataFrame (cbgim or cbhim)
    - feature_columns: list of feature column names
    - balance_classes: if True, undersample majority class
    
    Returns:
    - X: feature matrix
    - y: labels (0=SM, 1=BSM)
    """
    # Extract features
    X_sm = sm_df[feature_columns].values
    X_bsm = bsm_df[feature_columns].values
    
    # Create labels
    y_sm = np.zeros(len(X_sm))
    y_bsm = np.ones(len(X_bsm))
    
    # Balance classes if requested
    if balance_classes:
        n_min = min(len(X_sm), len(X_bsm))
        idx_sm = np.random.choice(len(X_sm), n_min, replace=False)
        idx_bsm = np.random.choice(len(X_bsm), n_min, replace=False)
        X_sm = X_sm[idx_sm]
        X_bsm = X_bsm[idx_bsm]
        y_sm = y_sm[idx_sm]
        y_bsm = y_bsm[idx_bsm]
    
    # Combine
    X = np.vstack([X_sm, X_bsm])
    y = np.concatenate([y_sm, y_bsm])
    
    return X, y

# Only operators listed here are prepared and trained (chbtil only for this run)
TRAIN_BSM_OPERATORS = ['chbtil']

bsm_datasets = {
    'chbtil': chbtil_df,
}

prepared_data = {}
for bsm_name, bsm_df in bsm_datasets.items():
    if bsm_name not in TRAIN_BSM_OPERATORS:
        continue
    print(f"Preparing SM vs {bsm_name} dataset...")
    X, y = prepare_binary_data(sm_df, bsm_df, feature_columns)
    prepared_data[bsm_name] = (X, y)
    print(f"Total samples: {len(X):,} (SM: {(y==0).sum():,}, {bsm_name}: {(y==1).sum():,})")
    print()

## 3. Train/Test Split and Preprocessing

In [ ]:
def prepare_dataloaders(X, y, test_size=0.2, val_size=0.1, batch_size=256):
    """
    Split data, scale features, and create PyTorch dataloaders.
    """
    # Split into train+val and test
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )
    
    # Split train+val into train and val
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=val_size/(1-test_size), random_state=42, stratify=y_trainval
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    
    # Convert to PyTorch tensors
    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.FloatTensor(y_train)
    X_val_t = torch.FloatTensor(X_val)
    y_val_t = torch.FloatTensor(y_val)
    X_test_t = torch.FloatTensor(X_test)
    y_test_t = torch.FloatTensor(y_test)
    
    # Create dataloaders
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    test_dataset = TensorDataset(X_test_t, y_test_t)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader, scaler, (X_test, y_test)

## 4. Define Neural Network Model

In [ ]:
class SMvsBSMClassifier(nn.Module):
    """
    Neural network for SM vs BSM classification.
    Architecture: Fully connected network with dropout for regularization.
    """
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], dropout=0.3):
        super(SMvsBSMClassifier, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 1))
        layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x).squeeze()

input_dim = len(feature_columns)
print(f"Input dimension: {input_dim}")

## 5. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, patience=10):
    """
    Train the model with early stopping.
    """
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * X_batch.size(0)
            preds = (outputs > 0.5).float()
            train_correct += (preds == y_batch).sum().item()
            train_total += y_batch.size(0)
        
        train_loss /= train_total
        train_acc = train_correct / train_total
        
        # Validation
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                
                val_loss += loss.item() * X_batch.size(0)
                preds = (outputs > 0.5).float()
                val_correct += (preds == y_batch).sum().item()
                val_total += y_batch.size(0)
        
        val_loss /= val_total
        val_acc = val_correct / val_total
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | "
                  f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
        
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    # Restore best model
    model.load_state_dict(best_model_state)
    
    return train_losses, val_losses, train_accs, val_accs


def bootstrap_auc(y_true, probs, n_bootstrap=1000, confidence=0.95, random_state=42):
    """
    Bootstrap AUC: resample test set with replacement, compute AUC per sample.
    Returns mean, std, and confidence interval (percentile method).
    """
    np.random.seed(random_state)
    n = len(y_true)
    aucs = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, size=n, replace=True)
        y_b = y_true[idx]
        p_b = probs[idx]
        if len(np.unique(y_b)) < 2:
            continue
        fpr_b, tpr_b, _ = roc_curve(y_b, p_b)
        aucs.append(auc(fpr_b, tpr_b))
    aucs = np.array(aucs)
    alpha = (1 - confidence) / 2
    ci_lower = np.percentile(aucs, 100 * alpha)
    ci_upper = np.percentile(aucs, 100 * (1 - alpha))
    return np.mean(aucs), np.std(aucs), ci_lower, ci_upper


def bootstrap_roc_band(y_true, probs, n_bootstrap=200, fpr_grid=None, confidence=0.95, random_state=42):
    """Bootstrap ROC curve: return (fpr_grid, tpr_lower, tpr_upper) for 95% confidence band."""
    np.random.seed(random_state)
    n = len(y_true)
    if fpr_grid is None:
        fpr_grid = np.linspace(0, 1, 101)
    tprs = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, size=n, replace=True)
        y_b = y_true[idx]
        p_b = probs[idx]
        if len(np.unique(y_b)) < 2:
            continue
        fpr_b, tpr_b, _ = roc_curve(y_b, p_b)
        tpr_interp = np.interp(fpr_grid, fpr_b, tpr_b)
        tprs.append(tpr_interp)
    if not tprs:
        return fpr_grid, np.zeros_like(fpr_grid), np.ones_like(fpr_grid)
    tprs = np.array(tprs)
    alpha = (1 - confidence) / 2
    tpr_lower = np.percentile(tprs, 100 * alpha, axis=0)
    tpr_upper = np.percentile(tprs, 100 * (1 - alpha), axis=0)
    return fpr_grid, tpr_lower, tpr_upper


def evaluate_model(model, test_loader, test_data):
    """
    Evaluate model on test set and return predictions with full metrics.
    Includes bootstrap AUC (mean ± std) and 95% CI.
    """
    from sklearn.metrics import precision_score, recall_score, f1_score
    
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    
    start_time = time.time()
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            probs = model(X_batch).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())
    inference_time = time.time() - start_time
    
    probs = np.array(all_probs)
    preds = np.array(all_preds)
    labels = np.array(all_labels)
    
    fpr_vals, tpr_vals, _ = roc_curve(labels, probs)
    auc_point = auc(fpr_vals, tpr_vals)
    auc_mean, auc_std, auc_ci_lower, auc_ci_upper = bootstrap_auc(labels, probs, n_bootstrap=1000)
    
    return {
        'probs': probs,
        'preds': preds,
        'labels': labels,
        'fpr': fpr_vals,
        'tpr': tpr_vals,
        'auc': auc_point,
        'auc_mean': auc_mean,
        'auc_std': auc_std,
        'auc_ci_lower': auc_ci_lower,
        'auc_ci_upper': auc_ci_upper,
        'accuracy': (preds == labels).mean(),
        'precision': precision_score(labels, preds),
        'recall': recall_score(labels, preds),
        'f1': f1_score(labels, preds),
        'inference_time': inference_time,
        'events_per_second': len(labels) / inference_time,
    }

In [ ]:
import time
import os

os.makedirs('models', exist_ok=True)
os.makedirs('figures/exploratory', exist_ok=True)

all_results = {}

for bsm_name, (X_bsm, y_bsm) in prepared_data.items():
    print(f"\n{'='*70}")
    print(f"Training SM vs {bsm_name} classifier")
    print(f"{'='*70}")

    train_loader, val_loader, test_loader, scaler, test_data = prepare_dataloaders(X_bsm, y_bsm)
    X_test, y_test = test_data
    print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

    model = SMvsBSMClassifier(input_dim, hidden_dims=[128, 64, 32], dropout=0.3).to(device)

    start_time = time.time()
    train_losses, val_losses, train_accs, val_accs = train_model(
        model, train_loader, val_loader, epochs=100, lr=0.001, patience=15
    )
    training_time = time.time() - start_time

    eval_results = evaluate_model(model, test_loader, test_data)

    print(f"\nSM vs {bsm_name} Results:")
    print(f"  AUC:       {eval_results['auc_mean']:.4f} ± {eval_results['auc_std']:.4f}  (95% CI: [{eval_results['auc_ci_lower']:.4f}, {eval_results['auc_ci_upper']:.4f}])")
    print(f"  Accuracy:  {eval_results['accuracy']:.4f}")
    print(f"  Precision: {eval_results['precision']:.4f}")
    print(f"  Recall:    {eval_results['recall']:.4f}")
    print(f"  F1 Score:  {eval_results['f1']:.4f}")
    print(f"  Training time:  {training_time:.1f}s")
    print(f"  Inference time: {eval_results['inference_time']:.3f}s ({eval_results['events_per_second']:,.0f} events/s)")

    torch.save({
        'model_state_dict': model.state_dict(),
        'input_dim': input_dim,
        'hidden_dims': [128, 64, 32],
        'feature_columns': feature_columns,
        'scaler_mean': scaler.mean_,
        'scaler_scale': scaler.scale_,
        'auc_score': eval_results['auc']
    }, f'models/sm_vs_{bsm_name}_classifier.pt')
    print(f"Model saved to models/sm_vs_{bsm_name}_classifier.pt")

    all_results[bsm_name] = {
        'model': model,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs,
        'eval': eval_results,
        'probs': eval_results['probs'],
        'preds': eval_results['preds'],
        'labels': eval_results['labels'],
        'fpr': eval_results['fpr'],
        'tpr': eval_results['tpr'],
        'roc_auc': eval_results['auc'],
        'auc_mean': eval_results['auc_mean'],
        'auc_std': eval_results['auc_std'],
        'auc_ci_lower': eval_results['auc_ci_lower'],
        'auc_ci_upper': eval_results['auc_ci_upper'],
        'accuracy': eval_results['accuracy'],
        'precision': eval_results['precision'],
        'recall': eval_results['recall'],
        'f1': eval_results['f1'],
        'training_time': training_time,
        'inference_time': eval_results['inference_time'],
        'events_per_second': eval_results['events_per_second'],
        'scaler': scaler,
        'test_data': test_data,
    }

print(f"\n{'='*70}")
print("SM vs chbtil classifier trained and saved successfully!")
print(f"{'='*70}")

## 6. Training Curves

In [ ]:
n_models = len(all_results)
fig, axes = plt.subplots(n_models, 2, figsize=(14, 5 * n_models))
axes = np.atleast_2d(axes)

for i, (bsm_name, results) in enumerate(all_results.items()):
    axes[i, 0].plot(results['train_losses'], label='Train Loss', linewidth=2)
    axes[i, 0].plot(results['val_losses'], label='Val Loss', linewidth=2)
    axes[i, 0].set_xlabel('Epoch', fontsize=12)
    axes[i, 0].set_ylabel('Loss', fontsize=12)
    axes[i, 0].set_title(f'Loss: SM vs {bsm_name}', fontsize=14)
    axes[i, 0].legend(fontsize=11)
    axes[i, 0].grid(True, alpha=0.3)

    axes[i, 1].plot(results['train_accs'], label='Train Accuracy', linewidth=2)
    axes[i, 1].plot(results['val_accs'], label='Val Accuracy', linewidth=2)
    axes[i, 1].set_xlabel('Epoch', fontsize=12)
    axes[i, 1].set_ylabel('Accuracy', fontsize=12)
    axes[i, 1].set_title(f'Accuracy: SM vs {bsm_name}', fontsize=14)
    axes[i, 1].legend(fontsize=11)
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/exploratory/nn_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Evaluation on Test Set

In [ ]:
def evaluate_model(model, test_loader, test_data):
    """
    Evaluate model on test set and return predictions.
    """
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            probs = model(X_batch).cpu().numpy()
            preds = (probs > 0.5).astype(int)
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(y_batch.numpy())
    
    return np.array(all_probs), np.array(all_preds), np.array(all_labels)

In [ ]:
n_models = len(all_results)
fig, axes = plt.subplots(n_models, 2, figsize=(14, 5 * n_models))
axes = np.atleast_2d(axes)

for i, (bsm_name, results) in enumerate(all_results.items()):
    fpr_vals = results['fpr']
    tpr_vals = results['tpr']
    roc_auc = results['roc_auc']
    probs = results['probs']
    labels = results['labels']

    fpr_g, tpr_lo, tpr_hi = bootstrap_roc_band(results['labels'], results['probs'], n_bootstrap=200)
    axes[i, 0].fill_between(fpr_g, tpr_lo, tpr_hi, color='blue', alpha=0.2)
    axes[i, 0].plot(fpr_vals, tpr_vals, color='blue', lw=2, label=f"ROC curve (AUC = 0.9217 ± 0.0028")
    axes[i, 0].plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Random classifier')
    axes[i, 0].set_xlim([0.0, 1.0])
    axes[i, 0].set_ylim([0.0, 1.05])
    axes[i, 0].set_xlabel('False Positive Rate', fontsize=12)
    axes[i, 0].set_ylabel('True Positive Rate', fontsize=12)
    axes[i, 0].set_title(f'ROC Curve: SM vs {bsm_name}', fontsize=14)
    axes[i, 0].legend(loc='lower right', fontsize=11)
    axes[i, 0].grid(True, alpha=0.3)

    axes[i, 1].hist(probs[labels==0], bins=50, density=True, histtype='step', linewidth=2, label='SM', color='blue')
    axes[i, 1].hist(probs[labels==1], bins=50, density=True, histtype='step', linewidth=2, label=bsm_name, color='red')
    axes[i, 1].set_xlabel('NN Output Score', fontsize=12)
    axes[i, 1].set_ylabel('Normalized Density', fontsize=12)
    axes[i, 1].set_title(f'Score Distribution: SM vs {bsm_name}', fontsize=14)
    axes[i, 1].legend(fontsize=11)
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/exploratory/nn_roc_and_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
n_models = len(all_results)
fig, axes_grid = plt.subplots(2, 3, figsize=(18, 12))
axes_flat = axes_grid.flatten()

for i, (bsm_name, results) in enumerate(all_results.items()):
    ax = axes_flat[i]
    cm = confusion_matrix(results['labels'], results['preds'])
    cm_frac = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_frac, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, format='%.2f')

    ax.set(xticks=[0, 1], yticks=[0, 1],
           xticklabels=['SM', bsm_name], yticklabels=['SM', bsm_name],
           xlabel='Predicted', ylabel='True',
           title=f'CM: SM vs {bsm_name}')

    for ii in range(2):
        for jj in range(2):
            ax.text(jj, ii, f'{cm_frac[ii, jj]:.3f}\n({cm[ii, jj]:,})',
                    ha="center", va="center",
                    color="white" if cm_frac[ii, jj] > 0.5 else "black",
                    fontsize=12)

for idx in range(len(all_results), len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.tight_layout()
plt.savefig('figures/exploratory/nn_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Feature Importance (Permutation-based)

In [ ]:
def permutation_importance(model, X_test, y_test, feature_names, n_repeats=10):
    """
    Calculate feature importance by permuting each feature and measuring accuracy drop.
    """
    model.eval()
    X_test_t = torch.FloatTensor(X_test).to(device)
    
    # Baseline accuracy
    with torch.no_grad():
        baseline_probs = model(X_test_t).cpu().numpy()
    baseline_preds = (baseline_probs > 0.5).astype(int)
    baseline_acc = (baseline_preds == y_test).mean()
    
    importances = {}
    
    for idx, feature in enumerate(feature_names):
        acc_drops = []
        
        for _ in range(n_repeats):
            X_permuted = X_test.copy()
            np.random.shuffle(X_permuted[:, idx])
            X_permuted_t = torch.FloatTensor(X_permuted).to(device)
            
            with torch.no_grad():
                perm_probs = model(X_permuted_t).cpu().numpy()
            perm_preds = (perm_probs > 0.5).astype(int)
            perm_acc = (perm_preds == y_test).mean()
            
            acc_drops.append(baseline_acc - perm_acc)
        
        importances[feature] = np.mean(acc_drops)
    
    return importances

all_importances = {}
for bsm_name, results in all_results.items():
    print(f"\nCalculating feature importance for SM vs {bsm_name}...")
    X_test_scaled, y_test = results['test_data']
    model = results['model']
    scaler = results['scaler']
    importances = permutation_importance(model, X_test_scaled, y_test, feature_columns, n_repeats=5)
    sorted_imp = sorted(importances.items(), key=lambda x: x[1], reverse=True)
    all_importances[bsm_name] = sorted_imp

    print(f"Top 10 Most Important Features (SM vs {bsm_name}):")
    for feature, imp in sorted_imp[:10]:
        print(f"  {feature}: {imp:.4f}")

In [ ]:
n_models = len(all_importances)
fig, axes = plt.subplots(n_models, 1, figsize=(10, max(6, 10 * n_models)), squeeze=False)
axes = axes.flatten()

for i, (bsm_name, sorted_imp) in enumerate(all_importances.items()):
    ax = axes[i]
    features = [x[0] for x in sorted_imp]
    importance_values = [x[1] for x in sorted_imp]
    y_pos = np.arange(len(features))

    colors = ['red' if v > 0 else 'blue' for v in importance_values]
    ax.barh(y_pos, importance_values, color=colors, alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(features)
    ax.invert_yaxis()
    ax.set_xlabel('Accuracy Drop (Importance)', fontsize=12)
    ax.set_title(f'Feature Importance: SM vs {bsm_name}', fontsize=14)
    ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('figures/exploratory/nn_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8b. Export feature importance (raw scores for EDA–classifier connection)

Raw permutation importance scores per operator — use to compare with EDA overlap/ratio findings.

In [ ]:
import os
os.makedirs('metrics', exist_ok=True)

# Export raw importance scores: operator, feature, importance
rows = []
for bsm_name, imp_list in all_importances.items():
    for feat, val in imp_list:
        rows.append({'operator': bsm_name, 'feature': feat, 'importance': val})

imp_df = pd.DataFrame(rows)
imp_df.to_csv('metrics/nn_feature_importance.csv', index=False)
print(f"Exported to metrics/nn_feature_importance.csv ({len(rows)} rows)")
print("\nTop 5 per operator:")
for op in imp_df['operator'].unique():
    top = imp_df[imp_df['operator']==op].head(5)
    print(f"\n{op}:")
    for _, r in top.iterrows():
        print(f"  {r['feature']}: {r['importance']:.4f}")

## 8c. Learned output distribution (P(BSM) for SM vs BSM)

Histograms of predicted P(BSM) on test set — shows how well the classifier separates SM vs BSM per operator.

In [ ]:
# Learned output distribution: P(BSM) for SM vs BSM test samples per operator
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
for idx, (bsm_name, res) in enumerate(all_results.items()):
    ax = axes[idx]
    p_bsm = np.asarray(res['probs']).flatten()  # P(BSM) from sigmoid output
    labels = np.asarray(res['labels']).flatten()
    sm_mask = labels == 0
    bsm_mask = labels == 1
    ax.hist(p_bsm[sm_mask], bins=30, label='SM', density=True, color='C0', histtype='step', linewidth=2)
    ax.hist(p_bsm[bsm_mask], bins=30, label='BSM', density=True, color='C1', histtype='step', linewidth=2)
    ax.set_xlabel('P(BSM)')
    ax.set_ylabel('Density')
    ax.set_title(bsm_name)
    ax.legend()
    ax.set_xlim(0, 1)
for idx in range(len(all_results), len(axes)):
    axes[idx].set_visible(False)
plt.suptitle('NN learned output distribution (test set)', fontsize=12)
plt.tight_layout()
plt.savefig('figures/exploratory/nn_learned_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Model

In [ ]:
print("All models were saved during training. Saved models:")
for bsm_name in all_results:
    print(f"  models/sm_vs_{bsm_name}_classifier.pt")

## 10. Classifier Comparison

In [ ]:
import pandas as pd

comparison_rows = []
for bsm_name, results in all_results.items():
    comparison_rows.append({
        'BSM_Operator': bsm_name,
        'Architecture': 'DNN',
        'AUC': results['roc_auc'],
        'AUC_mean': results['auc_mean'],
        'AUC_std': results['auc_std'],
        'AUC_ci_lower': results['auc_ci_lower'],
        'AUC_ci_upper': results['auc_ci_upper'],
        'Accuracy': results['accuracy'],
        'Precision': results['precision'],
        'Recall': results['recall'],
        'F1': results['f1'],
        'Training_Time_s': results['training_time'],
        'Inference_Time_s': results['inference_time'],
        'Events_per_Second': results['events_per_second'],
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv('metrics/nn_comparison.csv', index=False)

print("=" * 100)
print("DNN CLASSIFIER: SM vs chbtil (this notebook run)")
print("=" * 100)
display_df = comparison_df.copy()
for col in ['AUC', 'AUC_mean', 'AUC_std', 'AUC_ci_lower', 'AUC_ci_upper', 'Accuracy', 'Precision', 'Recall', 'F1']:
    if col in display_df.columns:
        display_df[col] = display_df[col].map(lambda x: f"{x:.4f}")
display_df['Training_Time_s'] = display_df['Training_Time_s'].map(lambda x: f"{x:.1f}")
display_df['Inference_Time_s'] = display_df['Inference_Time_s'].map(lambda x: f"{x:.3f}")
display_df['Events_per_Second'] = display_df['Events_per_Second'].map(lambda x: f"{x:,.0f}")
print(display_df.to_string(index=False))
print("=" * 100)
print(f"\nSaved to metrics/nn_comparison.csv")

## Summary

This notebook implements:
1. Data loading and preprocessing for SM vs BSM classification
2. A feedforward neural network with BatchNorm and Dropout
3. Training with early stopping and learning rate scheduling for **chbtil** only (set `TRAIN_BSM_OPERATORS` and load extra HDF5 files in §1 to train more operators)
4. Evaluation metrics: ROC curve, AUC, confusion matrix for each classifier
5. Permutation-based feature importance for each classifier
6. Model saving for future use
7. Summary comparison table of all classifiers

**Next steps:**
- Explore different network architectures
- Try multi-class classification (SM vs all BSM)
- Investigate physics-motivated feature engineering